# V7 数据获取测试 Notebook

本 notebook 系统测试 V7 系统所有数据源的数据获取能力，覆盖：

| 数据源 | 类型 | 测试内容 |
|--------|------|----------|
| TDX 标准连接 | 指数/股票 | 沪深300、中证500 等 |
| TDX 扩展连接 | 期货/期权 | IF/IH/IC/IM、商品期货 |
| PostgreSQL tdxIndex | 合约映射 | 历史数据回退 |
| PostgreSQL csiIndexPE | PE/PB | 估值百分位数据 |
| AKShare | 外盘期货 | WTI、COMEX黄金等 |
| pytdx 直连 | 期权PCR | 真实PCR数据 |
| 动态合约推导 | ContractManager | 近月/远月/期权合约 |


In [ ]:
import sys, os
from pathlib import Path

# 设置项目根目录
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f'项目根目录: {PROJECT_ROOT}')

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.4f}'.format)

print(f'pandas={pd.__version__}, numpy={np.__version__}')
print(f'当前时间: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

In [ ]:
# 加载 V7 系统配置
import yaml

config_path = PROJECT_ROOT / 'config' / 'market_state_system' / 'system_config.yaml'
if config_path.exists():
    with open(config_path, 'r', encoding='utf-8') as f:
        config = yaml.safe_load(f)
    print(f'✅ 配置加载成功: {config_path}')
    print(f'  系统版本: {config.get("system", {}).get("version")}')
    print(f'  TDX标准: {config.get("tdx", {}).get("hq_host")}:{config.get("tdx", {}).get("hq_port")}')
    print(f'  TDX扩展: {config.get("tdx", {}).get("exhq_host")}:{config.get("tdx", {}).get("exhq_port")}')
else:
    print(f'⚠️ 配置文件不存在: {config_path}')
    config = {}

## 1. TDX 标准连接测试（指数/股票）

TDX 标准连接 (hq_host:180.153.18.170:7709) 用于获取 A股/指数数据

In [ ]:
# 初始化 TDXAdapter — 标准连接
from data_service.tdx_adapter import TDXAdapter

tdx_standard = TDXAdapter({
    'host': config.get('tdx', {}).get('hq_host', '180.153.18.170'),
    'port': config.get('tdx', {}).get('hq_port', 7709),
    'pool_size': 3,
    'retry_count': 2,
})
print(f'TDX 标准连接状态: {"✅可用" if tdx_standard.is_available() else "❌不可用"}')
print(repr(tdx_standard))

In [ ]:
# 测试获取指数数据 — 沪深300
if tdx_standard.is_available():
    df_hs300 = tdx_standard.get_index_daily('000300', 'index_sh', days=500)
    if df_hs300 is not None and len(df_hs300) > 0:
        print(f'✅ 沪深300 数据获取成功: {len(df_hs300)} 条')
        print(f'  日期范围: {df_hs300["datetime"].min()} ~ {df_hs300["datetime"].max()}')
        print(f'  列名: {list(df_hs300.columns)}')
        display(df_hs300.tail(5))
    else:
        print('❌ 沪深300 数据获取失败')
else:
    print('⚠️ TDX 标准连接不可用，跳过')

In [ ]:
# 测试获取中证500、中证1000、中证2000
test_indices = {
    '000300': ('沪深300', 'index_sh'),
    '000905': ('中证500', 'index_sh'),
    '000852': ('中证1000', 'index_sh'),
    '932000': ('中证2000', 'index_sh'),
}

index_results = {}
if tdx_standard.is_available():
    for code, (name, mtype) in test_indices.items():
        df = tdx_standard.get_index_daily(code, mtype, days=60)
        if df is not None and len(df) > 0:
            index_results[name] = {'code': code, 'rows': len(df), 'latest': df['close'].iloc[-1]}
            print(f'✅ {name}({code}): {len(df)}条 | 最新收盘={df["close"].iloc[-1]:.2f}')
        else:
            print(f'❌ {name}({code}): 获取失败')
else:
    print('⚠️ TDX 不可用')

pd.DataFrame(index_results).T if index_results else print('无结果')

## 2. TDX 扩展连接测试（期货/期权）

TDX 扩展连接 (exhq_host:47.112.95.207:7720) 用于获取期货、期权等衍生品数据

In [ ]:
# 初始化 TDXAdapter — 扩展连接
tdx_extended = TDXAdapter({
    'host': config.get('tdx', {}).get('exhq_host', '47.112.95.207'),
    'port': config.get('tdx', {}).get('exhq_port', 7720),
    'pool_size': 3,
    'retry_count': 2,
})
print(f'TDX 扩展连接状态: {"✅可用" if tdx_extended.is_available() else "❌不可用"}')
print(repr(tdx_extended))

In [ ]:
# 测试获取股指期货数据
from market_state_system.core.contract_manager import ContractManager

cm = ContractManager(reference_date=datetime.now())
idx_futures = cm.get_index_futures_contracts()

futures_results = {}
if tdx_extended.is_available():
    for key, contract in idx_futures.items():
        # 先尝试主力合约(主连)
        main_code = cm.get_index_futures_main_code(key)
        df = tdx_extended.get_future_daily(main_code, 'future_zz', days=60)
        if df is not None and len(df) > 0:
            futures_results[key] = {
                'main_code': main_code,
                'active_code': contract.futures_code,
                'rows': len(df),
                'latest_close': df['close'].iloc[-1],
            }
            print(f'✅ {key.upper()}({main_code}): {len(df)}条 | 收盘={df["close"].iloc[-1]:.2f}')
        else:
            print(f'❌ {key.upper()}({main_code}): 获取失败')
else:
    print('⚠️ TDX 扩展连接不可用')

pd.DataFrame(futures_results).T if futures_results else print('无结果')

In [ ]:
# 测试获取商品期货数据
commodity_results = {}
if tdx_extended.is_available():
    commodity_pairs = cm.get_commodity_contracts()
    for key, pair in commodity_pairs.items():
        # 尝试主力合约(主连)
        main_code = f'{pair.variety_code}L8'
        df = tdx_extended.get_future_daily(main_code, pair.market_type, days=30)
        if df is not None and len(df) > 0:
            commodity_results[key] = {
                'variety': pair.variety_code,
                'main_code': main_code,
                'near_code': pair.near_code,
                'far_code': pair.far_code,
                'market_type': pair.market_type,
                'rows': len(df),
                'latest_close': df['close'].iloc[-1],
            }
            print(f'✅ {key}({main_code}): {len(df)}条 | 收盘={df["close"].iloc[-1]:.2f}')
        else:
            print(f'❌ {key}({main_code}): 获取失败')
else:
    print('⚠️ TDX 扩展连接不可用')

pd.DataFrame(commodity_results).T if commodity_results else print('无结果')

## 3. 期权 PCR 数据测试（pytdx 直连）

V7 核心改进：使用 pytdx 直连扩展行情获取真实期权持仓量数据，
而非 V6 的随机数伪造。PCR (Put-Call Ratio) = 看跌持仓量 / 看涨持仓量

In [ ]:
# 期权PCR数据获取 — 使用pytdx扩展行情直连
def fetch_option_pcr_simple(tdx_adapter, underlying='IO', market_code=7, near_month=None):
    """
    简化版期权PCR获取：
    1. 获取指定标的的期权合约列表
    2. 获取每个合约的日线数据(含持仓量)
    3. 按看涨/看跌分类，计算PCR
    
    Args:
        tdx_adapter: TDXAdapter实例(扩展连接)
        underlying: 标的代码，如 'IO', '510300'
        market_code: TDX市场代码
        near_month: 近月合约月份(YYYYMM)，如 202606
    """
    if not tdx_adapter.is_available():
        return None
    
    # 获取期权合约列表
    from pytdx.hq import TdxHq_API
    conn = tdx_adapter._pool.acquire()
    if conn is None:
        return None
    
    try:
        # 获取该市场的合约列表
        all_options = []
        start = 0
        while True:
            batch = conn.get_security_list(market_code, start)
            if not batch:
                break
            for item in batch:
                code = item.get('code', '')
                name = item.get('name', '')
                # 筛选目标标的的期权
                if underlying == 'IO' and code.startswith('IO'):
                    all_options.append({'code': code, 'name': name})
                elif underlying == 'MO' and code.startswith('MO'):
                    all_options.append({'code': code, 'name': name})
                elif underlying in code:
                    all_options.append({'code': code, 'name': name})
            start += len(batch)
            if len(batch) < 1000:
                break
        
        tdx_adapter._pool.release(conn)
        
        # 分类看涨/看跌
        calls = [o for o in all_options if '-C-' in o['name'] or 'C3' in o['name']]
        puts = [o for o in all_options if '-P-' in o['name'] or 'P3' in o['name']]
        
        print(f'  {underlying} 期权合约: 总={len(all_options)}, 看涨={len(calls)}, 看跌={len(puts)}')
        
        # 获取看涨持仓量总和
        call_oi = 0
        for opt in calls[:20]:  # 限制数量避免超时
            df = tdx_adapter.get_option_daily(opt['code'], 'option_zj', days=1)
            if df is not None and len(df) > 0 and 'open_interest' in df.columns:
                call_oi += float(df['open_interest'].iloc[-1])
        
        # 获取看跌持仓量总和
        put_oi = 0
        for opt in puts[:20]:
            df = tdx_adapter.get_option_daily(opt['code'], 'option_zj', days=1)
            if df is not None and len(df) > 0 and 'open_interest' in df.columns:
                put_oi += float(df['open_interest'].iloc[-1])
        
        pcr = put_oi / call_oi if call_oi > 0 else None
        
        return {
            'underlying': underlying,
            'total_options': len(all_options),
            'call_count': len(calls),
            'put_count': len(puts),
            'call_oi': call_oi,
            'put_oi': put_oi,
            'pcr': round(pcr, 4) if pcr else None,
        }
    except Exception as e:
        print(f'❌ PCR获取异常: {e}')
        return None

print('期权PCR获取函数已定义')

In [ ]:
# 执行PCR数据获取测试
if tdx_extended.is_available():
    pcr_result = fetch_option_pcr_simple(tdx_extended, underlying='IO')
    if pcr_result:
        print(f'\n✅ IO(沪深300)期权PCR数据:')
        print(f'  合约总数: {pcr_result["total_options"]}')
        print(f'  看涨合约: {pcr_result["call_count"]}')
        print(f'  看跌合约: {pcr_result["put_count"]}')
        print(f'  看涨持仓量: {pcr_result["call_oi"]:.0f}')
        print(f'  看跌持仓量: {pcr_result["put_oi"]:.0f}')
        print(f'  PCR(看跌/看涨): {pcr_result["pcr"]}')
    else:
        print('❌ PCR获取失败')
else:
    print('⚠️ TDX 扩展连接不可用，跳过PCR测试')

## 4. PostgreSQL 数据库测试

V7 使用两个 PostgreSQL 数据库：
- `tdxIndex` (10.3.18.56): 指数历史数据、合约映射
- `csiIndexPE`: PE/PB 估值数据

In [ ]:
# 初始化 DatabaseReader
from data_service.database_reader import DatabaseReader
from config.global_settings import DATABASE_ENGINES, DB_POOL_CONFIG

print('数据库配置:')
for name, url in DATABASE_ENGINES.items():
    # 隐藏密码
    safe_url = url.split('@')[0].rsplit(':', 1)[0] + ':***@' + url.split('@')[-1] if '@' in url else url
    print(f'  {name}: {safe_url}')

db_reader = DatabaseReader(DATABASE_ENGINES, DB_POOL_CONFIG)
print(f'\nDatabaseReader 初始化完成, 引擎数: {len(db_reader._engines)}')

In [ ]:
# 测试 tdxIndex 数据库
print('=== tdxIndex 数据库健康检查 ===')
try:
    is_healthy = db_reader.health_check('index_db')
    print(f'  健康状态: {"✅" if is_healthy else "❌"}')
    
    if is_healthy:
        # 尝试读取沪深300数据
        df_db = db_reader.read_table('000300', engine_name='index_db',
                                     order_by='datetime DESC', limit=5,
                                     parse_dates=['datetime'])
        if not df_db.empty:
            print(f'  ✅ 沪深300数据: {len(df_db)}条')
            display(df_db.head())
        else:
            print('  ⚠️ 无数据返回')
except Exception as e:
    print(f'❌ tdxIndex测试失败: {e}')

In [ ]:
# 测试 csiIndexPE 数据库
print('=== csiIndexPE 数据库健康检查 ===')
try:
    is_healthy = db_reader.health_check('index_pe_db')
    print(f'  健康状态: {"✅" if is_healthy else "❌"}')
    
    if is_healthy:
        # 尝试读取沪深300 PE数据
        df_pe = db_reader.read_table('000300', engine_name='index_pe_db',
                                     order_by='date DESC', limit=5,
                                     parse_dates=['date'])
        if not df_pe.empty:
            print(f'  ✅ 沪深300 PE数据: {len(df_pe)}条')
            print(f'  列名: {list(df_pe.columns)}')
            display(df_pe.head())
        else:
            print('  ⚠️ 无PE数据返回')
except Exception as e:
    print(f'❌ csiIndexPE测试失败: {e}')

## 5. AKShare 外盘数据测试

AKShare 适配器用于获取外盘期货、汇率等数据

In [ ]:
# 初始化 AKAdapter
from data_service.ak_adapter import AKAdapter

ak = AKAdapter({'cache_enabled': True, 'cache_ttl': 300})
print(f'AKAdapter 状态: {"✅可用" if ak.is_available() else "❌不可用"}')
print(repr(ak))

In [ ]:
# 测试外盘期货数据
test_symbols = ['CL', 'GC', 'SI', 'DX']  # WTI原油、COMEX黄金、COMEX白银、美元指数

ak_results = {}
if ak.is_available():
    for symbol in test_symbols:
        result = ak.get_futures_realtime(symbol)
        if result:
            ak_results[symbol] = result
            print(f'✅ {symbol}({result["name"]}): 价格={result["price"]} {result.get("price_unit", "")} 涨跌={result.get("change_pct", 0):+.2f}%')
        else:
            print(f'❌ {symbol}: 获取失败')
else:
    print('⚠️ AKShare 不可用')

pd.DataFrame(ak_results).T if ak_results else print('无结果')

## 6. ContractManager 动态合约推导测试

V7 核心改进：基于当前日期动态推导期货/期权合约代码，零硬编码

In [ ]:
# 初始化 ContractManager
code_table_path = str(PROJECT_ROOT / 'data' / 'tdx_code_table.xlsx')
cm = ContractManager(
    code_table_path=code_table_path if os.path.exists(code_table_path) else None,
    reference_date=datetime.now(),
    rollover_day=config.get('contract_manager', {}).get('rollover_day', 15),
)
print(f'ContractManager 初始化完成')
print(f'  参考日期: {cm.reference_date.strftime("%Y-%m-%d")}')
print(f'  滚动日: 每月{cm.rollover_day}号后切换')
print(f'  已加载合约数: {len(cm._code_table)}')

In [ ]:
# 测试商品期货合约推导
commodity_pairs = cm.get_commodity_contracts()
print(f'商品期货合约推导结果 ({len(commodity_pairs)}个品种):')
print(f'{"品种":<12} {"代码":<6} {"近月合约":<10} {"远月合约":<10} {"近月交割":<12} {"市场类型"}')
print('-' * 70)
for key, pair in commodity_pairs.items():
    print(f'{key:<12} {pair.variety_code:<6} {pair.near_code:<10} {pair.far_code:<10} '
          f'{pair.near_year}-{pair.near_month:02d}/{pair.far_year}-{pair.far_month:02d}  {pair.market_type}')

In [ ]:
# 测试股指期货合约推导
idx_futures = cm.get_index_futures_contracts()
print(f'股指期货合约推导结果:')
print(f'{"品种":<6} {"当月合约":<10} {"下季月合约":<12} {"现货代码":<10} {"交割月"}')
print('-' * 55)
for key, contract in idx_futures.items():
    print(f'{key.upper():<6} {contract.futures_code:<10} {contract.next_quarter_code:<12} '
          f'{contract.spot_code:<10} {contract.delivery_year}-{contract.delivery_month:02d}')

In [ ]:
# 测试期权近月推导
option_underlyings = ['IO', 'MO', '510300', '588000']
print(f'期权近月交割推导:')
for underlying in option_underlyings:
    near_year, near_month = cm.get_option_near_month(underlying)
    print(f'  {underlying}: 近月 = {near_year}年{near_month}月')

## 7. DataLoadingService 集成测试

测试统一数据加载服务，验证 TDX → DB 降级逻辑

In [ ]:
# 初始化完整数据服务
from base_services.config_service import ConfigService
from base_services.cache_service import CacheService
from data_service.data_loader_service import DataLoadingService

# 简易配置服务
cfg_svc = ConfigService(system_name='market_state_system', enable_hot_reload=False)

# 初始化 DataLoadingService
data_svc = DataLoadingService(
    config_service=cfg_svc,
    database_reader=db_reader,
    tdx_adapter=tdx_standard,  # 标准连接
    ak_adapter=ak,
    cache_service=CacheService(max_size=100, ttl=3600),
    enable_cache=True,
)
print(f'DataLoadingService 初始化完成')

In [ ]:
# 测试各类数据加载
print('=== DataLoadingService 集成测试 ===\n')

# 1. 指数数据
df_idx = data_svc.load_index_data('000300', min_days=60)
print(f'1. 沪深300指数: {"✅" if len(df_idx) > 0 else "❌"} {len(df_idx)}条')

# 2. 衍生品数据
df_deriv = data_svc.load_derivative_data('IFL8', 'future_zj', days=30)
print(f'2. IF主连期货: {"✅" if len(df_deriv) > 0 else "❌"} {len(df_deriv)}条')

# 3. PE数据
df_pe = data_svc.load_pe_data('000300')
print(f'3. 沪深300 PE: {"✅" if len(df_pe) > 0 else "❌"} {len(df_pe)}条')

# 4. 宏观数据
macro_results = data_svc.load_all_macro_indicators()
print(f'4. 宏观指标: {len(macro_results)}个指标')
for k, v in macro_results.items():
    print(f'   {k}: {v}')

# 缓存统计
stats = data_svc.get_cache_stats()
print(f'\n缓存统计: 命中率={stats["hit_rate"]:.1%} | 使用={stats["size"]}')

## 8. 数据获取能力汇总

汇总所有数据源的可用性和数据质量

In [ ]:
# 数据获取能力汇总
summary = {
    '数据源': [],
    '类型': [],
    '状态': [],
    '数据量': [],
    '备注': [],
}

# TDX标准
summary['数据源'].append('TDX标准(7709)')
summary['类型'].append('指数/股票')
summary['状态'].append('✅' if tdx_standard.is_available() else '❌')
summary['数据量'].append(f'{len(df_idx)}条' if len(df_idx) > 0 else '0')
summary['备注'].append('沪深300等指数数据')

# TDX扩展
summary['数据源'].append('TDX扩展(7720)')
summary['类型'].append('期货/期权')
summary['状态'].append('✅' if tdx_extended.is_available() else '❌')
summary['数据量'].append(f'{len(futures_results)}品种' if futures_results else '0')
summary['备注'].append('IF/IH/IC/IM + 商品期货')

# PostgreSQL tdxIndex
summary['数据源'].append('PostgreSQL tdxIndex')
summary['类型'].append('历史数据')
try:
    db_ok = db_reader.health_check('index_db')
except:
    db_ok = False
summary['状态'].append('✅' if db_ok else '❌')
summary['数据量'].append('回退用')
summary['备注'].append('TDX不可用时降级')

# PostgreSQL csiIndexPE
summary['数据源'].append('PostgreSQL csiIndexPE')
summary['类型'].append('PE/PB')
try:
    pe_ok = db_reader.health_check('index_pe_db')
except:
    pe_ok = False
summary['状态'].append('✅' if pe_ok else '❌')
summary['数据量'].append(f'{len(df_pe)}条' if len(df_pe) > 0 else '0')
summary['备注'].append('估值百分位关键数据')

# AKShare
summary['数据源'].append('AKShare')
summary['类型'].append('外盘/汇率')
summary['状态'].append('✅' if ak.is_available() else '❌')
summary['数据量'].append(f'{len(ak_results)}品种' if ak_results else '0')
summary['备注'].append('WTI/黄金/美元指数')

# 期权PCR
summary['数据源'].append('pytdx直连PCR')
summary['类型'].append('期权PCR')
summary['状态'].append('✅' if pcr_result and pcr_result.get('pcr') else '❌')
summary['数据量'].append(f'PCR={pcr_result["pcr"]}' if pcr_result and pcr_result.get('pcr') else 'N/A')
summary['备注'].append('V7核心改进: 真实PCR')

# ContractManager
summary['数据源'].append('ContractManager')
summary['类型'].append('动态合约')
summary['状态'].append('✅')
summary['数据量'].append(f'{len(commodity_pairs)}商品+{len(idx_futures)}股指')
summary['备注'].append('零硬编码日期驱动')

df_summary = pd.DataFrame(summary)
print('V7 数据获取能力汇总:')
display(df_summary)

# 可用率统计
available = sum(1 for s in summary['状态'] if '✅' in s)
total = len(summary['状态'])
print(f'\n数据源可用率: {available}/{total} = {available/total:.0%}')

In [ ]:
# 清理资源
tdx_standard.close()
tdx_extended.close()
db_reader.close()
ak.close()
print('✅ 所有数据源连接已关闭')